# Interactive Dash Web Application

Install Dash

In [ ]:
!pip install dash jupyter-dash

In [ ]:
!pip install pyngrok

In [ ]:
!npm install -g localtunnel
!npx localtunnel --port 8050

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 868ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧⠙⠹⠸⠼⠴your url is: https://brown-wings-crash.loca.lt


In [ ]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.graph_objects as go

# -------------------------------------------------------------------------
# 1. Load Pre-Analyzed Artifacts
# -------------------------------------------------------------------------
df_constructor_pace = pd.read_csv("f1_constructor_pace_summary.csv")
df_midfield_kings = pd.read_csv("f1_midfield_kings_summary.csv")

# Format driver names for display
df_midfield_kings["driver_name"] = (
    df_midfield_kings["forename"] + " " + df_midfield_kings["surname"]
)

# Options for multi-select dropdown
driver_options = [
    {"label": name, "value": name}
    for name in sorted(df_midfield_kings["driver_name"].unique())
]

# Initialize Dash App
app = dash.Dash(__name__, title="F1 Midfield King Explorer")

# -------------------------------------------------------------------------
# 2. Layout Definition (No Year Slicer)
# -------------------------------------------------------------------------
app.layout = html.Div(
    style={"fontFamily": "Segoe UI, Arial, sans-serif", "padding": "20px"},
    children=[
        html.H1(
            "🏎️ F1 Performance Analytics — Midfield King Explorer",
            style={"textAlign": "center", "color": "#1e1e1e"},
        ),
        html.P(
            "Interactive exploration of pure pace gaps, grid determinism, and DNF-adjusted position gains (2014–2026).",
            style={"textAlign": "center", "color": "#666666"},
        ),
        html.Hr(),
        # Streamlined Control Panel
        html.Div(
            style={
                "backgroundColor": "#f8f9fa",
                "padding": "20px",
                "borderRadius": "8px",
                "display": "flex",
                "justifyContent": "space-around",
                "alignItems": "center",
                "gap": "20px",
                "marginBottom": "25px",
                "border": "1px solid #e9ecef",
            },
            children=[
                # Minimum Clean Races Slider
                html.Div(
                    [
                        html.Label(
                            "Minimum Clean Races Filter:",
                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "8px",
                            },
                        ),
                        dcc.Slider(
                            id="min-races-slider",
                            min=10,
                            max=100,
                            step=5,
                            value=30,
                            marks={i: str(i) for i in range(10, 101, 15)},
                        ),
                    ],
                    style={"width": "45%", "minWidth": "300px"},
                ),
                # Interactive Multi-Select Driver Dropdown
                html.Div(
                    [
                        html.Label(
                            "Filter Specific Driver(s) / Compare Teammates:",
                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "8px",
                            },
                        ),
                        dcc.Dropdown(
                            id="driver-dropdown",
                            options=driver_options,
                            multi=True,
                            placeholder="Select drivers to compare...",
                            style={"backgroundColor": "#ffffff"},
                        ),
                    ],
                    style={"width": "45%", "minWidth": "300px"},
                ),
            ],
        ),
        # Tabs Container
        dcc.Tabs(
            id="analytics-tabs",
            value="tab-pace",
            children=[
                dcc.Tab(
                    label="Pace & Consistency Matrix",
                    value="tab-pace",
                    children=[
                        dcc.Graph(id="pace-matrix-graph", style={"height": "600px"})
                    ],
                ),
                dcc.Tab(
                    label="DNF-Adjusted Position Mobility",
                    value="tab-mobility",
                    children=[
                        dcc.Graph(id="mobility-graph", style={"height": "600px"})
                    ],
                ),
            ],
        ),
    ],
)

# -------------------------------------------------------------------------
# 3. Callback (Slider & Dropdown Inputs Only)
# -------------------------------------------------------------------------
@app.callback(
    [Output("pace-matrix-graph", "figure"), Output("mobility-graph", "figure")],
    [Input("min-races-slider", "value"), Input("driver-dropdown", "value")],
)
def update_dashboard(min_races, selected_drivers):
    filtered_kings = df_midfield_kings[
        df_midfield_kings["total_clean_races"] >= min_races
    ].copy()

    if selected_drivers and len(selected_drivers) > 0:
        mobility_data = filtered_kings[
            filtered_kings["driver_name"].isin(selected_drivers)
        ].sort_values(by="avg_positions_gained", ascending=True)
        chart_title = f"Net Position Gain Comparison: Selected Drivers ({len(selected_drivers)})"
    else:
        mobility_data = filtered_kings.head(10).sort_values(
            by="avg_positions_gained", ascending=True
        )
        chart_title = f"Top 10 Midfield Drivers by Net Position Gain (Min. {min_races} Clean Races)"

    # Figure 1: Pace Matrix
    fig_pace = go.Figure()
    for is_midfield, group_df in df_constructor_pace.groupby("is_midfield"):
        cohort_label = "Midfield Team" if is_midfield else "Top-Tier Team"
        marker_color = "dodgerblue" if is_midfield else "crimson"

        fig_pace.add_trace(
            go.Scatter(
                x=group_df["avg_pace_gap"],
                y=group_df["avg_consistency_std"],
                mode="markers+text",
                name=cohort_label,
                text=group_df["constructor_name"],
                textposition="top right",
                marker=dict(
                    size=12,
                    color=marker_color,
                    line=dict(width=1, color="DarkSlateGrey"),
                ),
                customdata=group_df[
                    ["constructor_name", "avg_pace_gap", "avg_consistency_std"]
                ],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    + "Cohort: "
                    + cohort_label
                    + "<br>"
                    + "Avg Pace Gap: +%{customdata[1]:.2f}s<br>"
                    + "Stint Variance (σ): %{customdata[2]:.2f}s"
                    + "<extra></extra>"
                ),
            )
        )

    fig_pace.update_layout(
        title="Constructor Pure Pace Gap vs. Stint Consistency (2014–2026)",
        xaxis_title="Average Pace Gap to Race Leader (seconds)",
        yaxis_title="Stint Variance (Standard Deviation in seconds)",
        template="plotly_white",
        legend_title="Constructor Cohort",
    )

    # Figure 2: Position Mobility Bar Chart
    fig_mobility = go.Figure()
    fig_mobility.add_trace(
        go.Bar(
            y=mobility_data["driver_name"],
            x=mobility_data["avg_positions_gained"],
            orientation="h",
            marker=dict(color="dodgerblue"),
            error_x=dict(
                type="data",
                array=mobility_data["std_positions_gained"] / 2,
                visible=True,
            ),
            customdata=mobility_data[
                [
                    "driver_name",
                    "avg_positions_gained",
                    "std_positions_gained",
                    "total_clean_races",
                ]
            ],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                + "Avg Net Positions Gained: +%{customdata[1]:.2f}<br>"
                + "Position Gain Variance (σ): ±%{customdata[2]:.2f}<br>"
                + "Total Clean Races Evaluated: %{customdata[3]}"
                + "<extra></extra>"
            ),
        )
    )
    fig_mobility.update_layout(
        title=chart_title,
        xaxis_title="Average Net Positions Gained per Race",
        yaxis_title="Driver",
        template="plotly_white",
    )

    return fig_pace, fig_mobility


# Execution Block
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8050, debug=False)